# 阶段 3：MiniMind Dense 现代结构

本 Notebook 是阶段 3 的主要学习入口。它读取 MiniMind 官方固定 revision 的源码，按一次 forward 的数据流理解现代 Decoder-only Transformer，并用缩小配置观察关键 Tensor。

本阶段只看 Dense 主线，不进入 MoE、SFT、DPO 和 RL。

## 学习路线

Dense 表示每个 token 都经过同一组稠密 FFN 参数，与按 token 路由到不同专家的 MoE 相对。

本 Notebook 使用以下符号：批量大小为 $B$，序列长度为 $T$，隐藏维度为 $D$，层数为 $L$，词表大小为 $V$，query head 数为 $H_{\mathrm q}$，key/value head 数为 $H_{\mathrm{kv}}$，单个 head 的维度为 $d_{\mathrm{head}}$。

阅读顺序固定为：

```text
MiniMindConfig
→ RMSNorm
→ Attention（QK-Norm、RoPE、GQA、SDPA、KV Cache）
→ FeedForward
→ MiniMindBlock
→ MiniMindModel
→ MiniMindForCausalLM
→ PretrainDataset
→ train_pretrain.py
```

每一节先说明输入、输出和作用，再展示官方源码。

In [1]:
# 定位项目根目录并加载固定 revision 的官方模型
import importlib.util
import inspect
import json
import sys
from pathlib import Path

import torch
from IPython.display import Markdown, display

from llm_learning.minimind.inspect import (
    MINIMIND_REVISION,
    inspect_minimind,
    load_official_model_module,
)

repo_root = Path.cwd()
if not (repo_root / "pyproject.toml").is_file():
    repo_root = repo_root.parent
source_dir = repo_root / "third_party" / "minimind"
model_module = load_official_model_module(source_dir)

def show_source(source: str) -> None:
    display(Markdown(f"```python\n{source.rstrip()}\n```"))

print("项目根目录:", repo_root)
print("MiniMind revision:", MINIMIND_REVISION)
print("PyTorch:", torch.__version__)

项目根目录: /home/user/LLM-learning
MiniMind revision: 89d674b8a517010f5561b6d8ab2dcbb58e2fb91b
PyTorch: 2.9.1


## 1. MiniMindConfig：先确定所有维度

配置对象本身不处理 Tensor。它确定 $D$、$L$、$H_{\mathrm q}$、$H_{\mathrm{kv}}$、$d_{\mathrm{head}}$、FFN 中间维度和 RoPE 参数。代码中的对应字段分别是 `hidden_size`、`num_hidden_layers`、`num_attention_heads`、`num_key_value_heads` 和 `head_dim`。

缩小配置使用 $D=256$、$L=4$、$H_{\mathrm q}=4$ 和 $H_{\mathrm{kv}}=2$。因此

$$
d_{\mathrm{head}}=\frac{D}{H_{\mathrm q}}=64,
$$

每两个 query heads 共享一组 key/value heads。

In [2]:
# 先读官方配置源码，再打印本次实验真正使用的维度
show_source(inspect.getsource(model_module.MiniMindConfig))

config = model_module.MiniMindConfig(
    hidden_size=256,
    num_hidden_layers=4,
    num_attention_heads=4,
    num_key_value_heads=2,
    vocab_size=6400,
    max_position_embeddings=128,
    flash_attn=True,
    use_moe=False,
    dropout=0.0,
)
for name in [
    "hidden_size",
    "num_hidden_layers",
    "num_attention_heads",
    "num_key_value_heads",
    "head_dim",
    "vocab_size",
    "max_position_embeddings",
    "hidden_act",
    "intermediate_size",
    "bos_token_id",
    "eos_token_id",
    "rms_norm_eps",
    "rope_theta",
    "tie_word_embeddings",
    "flash_attn",
    "dropout",
]:
    print(f"{name}: {getattr(config, name)}")

```python
class MiniMindConfig(PretrainedConfig):
    model_type = "minimind"
    def __init__(self, hidden_size=768, num_hidden_layers=8, use_moe=False, **kwargs):
        super().__init__(**kwargs)
        self.hidden_size = hidden_size
        self.num_hidden_layers = num_hidden_layers
        self.use_moe = use_moe
        self.dropout = kwargs.get("dropout", 0.0)
        self.vocab_size = kwargs.get("vocab_size", 6400)
        self.bos_token_id = kwargs.get("bos_token_id", 1)
        self.eos_token_id = kwargs.get("eos_token_id", 2)
        self.flash_attn = kwargs.get("flash_attn", True)
        self.num_attention_heads = kwargs.get("num_attention_heads", 8)
        self.num_key_value_heads = kwargs.get("num_key_value_heads", 4)
        self.head_dim = kwargs.get("head_dim", self.hidden_size // self.num_attention_heads)
        self.hidden_act = kwargs.get("hidden_act", 'silu')
        self.intermediate_size = kwargs.get("intermediate_size", math.ceil(hidden_size * math.pi / 64) * 64)
        self.max_position_embeddings = kwargs.get("max_position_embeddings", 32768)
        self.rms_norm_eps = kwargs.get("rms_norm_eps", 1e-6)
        self.rope_theta = kwargs.get("rope_theta", 1e6)
        self.tie_word_embeddings = kwargs.get("tie_word_embeddings", True)
        self.inference_rope_scaling = kwargs.get("inference_rope_scaling", False)
        self.rope_scaling = {
            "beta_fast": 32,
            "beta_slow": 1,
            "factor": 16,
            "original_max_position_embeddings": 2048,
            "attention_factor": 1.0,
            "type": "yarn"
        } if self.inference_rope_scaling else None
        ### MoE specific configs (ignored if use_moe = False)
        self.num_experts = kwargs.get("num_experts", 4)
        self.num_experts_per_tok = kwargs.get("num_experts_per_tok", 1)
        self.moe_intermediate_size = kwargs.get("moe_intermediate_size", self.intermediate_size)
        self.norm_topk_prob = kwargs.get("norm_topk_prob", True)
        self.router_aux_loss_coef = kwargs.get("router_aux_loss_coef", 5e-4)
```

hidden_size: 256
num_hidden_layers: 4
num_attention_heads: 4
num_key_value_heads: 2
head_dim: 64
vocab_size: 6400
max_position_embeddings: 128
hidden_act: silu
intermediate_size: 832
bos_token_id: 1
eos_token_id: 2
rms_norm_eps: 1e-06
rope_theta: 1000000.0
tie_word_embeddings: True
flash_attn: True
dropout: 0.0


### 其余需要理解用途的配置

`intermediate_size` 是 gated FFN 的升维宽度。官方默认按

$$
D_{\mathrm{ff}}=64\left\lceil\frac{\pi D}{64}\right\rceil
$$

计算，使宽度约为 $\pi D$ 并向上对齐到 64 的整数倍。当前 $D=256$，所以 $D_{\mathrm{ff}}=832$。

| 配置字段 | 当前值 | 用途 |
| --- | ---: | --- |
| `vocab_size` | `6400` | tokenizer 可产生 6400 种 token ID，取值范围为 `0`～`6399`；它不是序列长度 |
| `max_position_embeddings` | `128` | 本次缩小实验预计算 128 个位置的 RoPE；官方默认值是 `32768`，当前随机输入的实际 $T=8$ |
| `hidden_act` | `silu` | 门控分支使用 SiLU；结合 `gate_proj`、`up_proj` 的逐元素乘法，构成 SwiGLU 风格 FFN |
| `bos_token_id` | `1` | 标记序列开始；它是 tokenizer 词表中的一个特殊 token ID |
| `eos_token_id` | `2` | 标记序列结束，也可作为生成停止条件 |
| `rms_norm_eps` | $10^{-6}$ | 加在 RMSNorm 分母中，避免除零并改善数值稳定性 |
| `rope_theta` | $10^6$ | RoPE 计算旋转频率时使用的 base ，影响位置频率 |
| `tie_word_embeddings` | `True` | 让输入 token embedding 与输出 LM head 共用同一份权重 |
| `flash_attn` | `True` | 允许模型调用 PyTorch SDPA；具体 backend 不一定是 FlashAttention |
| `dropout` | `0.0` | 当前结构检查不随机丢弃 attention 或 residual 输出 |

权重绑定发生在 `MiniMindForCausalLM.__init__` 中：`model.embed_tokens.weight` 与 `lm_head.weight` 指向同一个 Parameter，因此减少一份大小为 $V\times D$ 的参数矩阵。

`MiniMindConfig` 继承 `PretrainedConfig`，用于接入 Transformers 的配置序列化、保存和加载。当前 `inference_rope_scaling=False`、`use_moe=False`，因此 YaRN scaling 与 MoE 专用字段不参与本次 Dense forward，留到相应阶段再学习。

## 2. RMSNorm：稳定子层输入的数值尺度

RMS 是 Root Mean Square（均方根）的缩写。对每个 token，RMSNorm 接收向量 $\mathbf{x}\in\mathbb{R}^{D}$，输出维度不变：

$$
\operatorname{RMSNorm}(\mathbf{x})
=\mathbf{g}\odot
\frac{\mathbf{x}}{\sqrt{\frac{1}{D}\sum_{i=1}^{D}x_i^2+\varepsilon}}.
$$

其中 $\mathbf{g}\in\mathbb{R}^{D}$ 是代码中 `self.weight` 对应的可学习缩放向量，$\odot$ 表示逐元素乘法，不是点积。因此整批输入与输出均属于 $\mathbb{R}^{B\times T\times D}$。MiniMind 在 attention 前、FFN 前以及所有 block 之后使用 RMSNorm。

与 TinyGPT 的 LayerNorm 相比，它省略了均值中心化。这里的 normalization 用于稳定数值尺度，与 dropout、weight decay 等 regularization 不是同一概念。Pre-Norm 只把归一化结果送入 attention 或 FFN 分支，不会用它覆盖残差主干。

In [3]:
# 阅读实现，并用一个小 Tensor 验证输入输出 shape
show_source(inspect.getsource(model_module.RMSNorm))

torch.manual_seed(2026)
hidden_states = torch.randn(2, 8, 256)
normalized = model_module.RMSNorm(256, eps=1e-6)(hidden_states)
print("input shape:", tuple(hidden_states.shape))
print("output shape:", tuple(normalized.shape))
print("第一个 token 归一化后的 RMS:", normalized[0, 0].pow(2).mean().sqrt().item())

```python
class RMSNorm(torch.nn.Module):
    def __init__(self, dim: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def norm(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x):
        return (self.weight * self.norm(x.float())).type_as(x)
```

input shape: (2, 8, 256)
output shape: (2, 8, 256)
第一个 token 归一化后的 RMS: 0.9999994039535522


## 3. Attention：阶段 3 的核心

输入 hidden states 记为 $\mathbf{X}\in\mathbb{R}^{B\times T\times D}$。线性投影并拆分 head 后：

$$
\mathbf{Q}\in\mathbb{R}^{B\times T\times H_{\mathrm q}\times d_{\mathrm{head}}},\qquad
\mathbf{K},\mathbf{V}\in\mathbb{R}^{B\times T\times H_{\mathrm{kv}}\times d_{\mathrm{head}}}.
$$

QK-Norm 先控制每个 head 的数值尺度，RoPE 再按位置 $t$ 旋转 query 和 key：$\mathbf{q}'_t=R_t\mathbf{q}_t$，$\mathbf{k}'_t=R_t\mathbf{k}_t$。GQA 在 attention 计算前用 `repeat_kv` 将较少的 key/value heads 映射到 query heads。历史 key/value 在复制前进入 cache，因此单个 cache Tensor 保持紧凑的 $B\times T\times H_{\mathrm{kv}}\times d_{\mathrm{head}}$ 形状。

SDPA 是 PyTorch 的 scaled dot-product attention 接口。令 $\mathbf{M}$ 为加性 attention mask，其核心计算为

$$
\operatorname{Attention}(\mathbf{Q},\mathbf{K},\mathbf{V})
=\operatorname{softmax}\!\left(
\frac{\mathbf{Q}\mathbf{K}^{\mathsf T}}{\sqrt{d_{\mathrm{head}}}}+\mathbf{M}
\right)\mathbf{V}.
$$

$\mathbf{M}$ 至少包含 causal mask：允许关注的位置加 $0$，未来位置加 $-\infty$。需要 padding 或其他结构约束时，还可以合并 padding/structural mask。本次随机输入没有 padding，因此实验只需要 causal mask；源码通过可选的 `attention_mask` 参数处理额外 mask。

该接口负责 score、mask、softmax 和 value 加权，并可按设备选择优化 kernel。配置中的 `flash_attn=True` 只允许调用 SDPA，不保证当前设备和输入一定选择 FlashAttention backend。源码中各投影层的 `bias=False` 表示不创建偏置 Parameter，计算效果等价于偏置 $\mathbf b=\mathbf 0$。

In [4]:
# 连同 RoPE 和 repeat_kv 一起读 Attention 官方实现
show_source(inspect.getsource(model_module.precompute_freqs_cis))
show_source(inspect.getsource(model_module.apply_rotary_pos_emb))
show_source(inspect.getsource(model_module.repeat_kv))
show_source(inspect.getsource(model_module.Attention))

```python
def precompute_freqs_cis(dim: int, end: int = int(32 * 1024), rope_base: float = 1e6, rope_scaling: dict = None):
    freqs, attn_factor = 1.0 / (rope_base ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim)), 1.0
    if rope_scaling is not None: # YaRN: f'(i) = f(i)((1-γ) + γ/s), where γ∈[0,1] is linear ramp
        orig_max, factor, beta_fast, beta_slow, attn_factor = (
            rope_scaling.get("original_max_position_embeddings", 2048), rope_scaling.get("factor", 16),
            rope_scaling.get("beta_fast", 32.0), rope_scaling.get("beta_slow", 1.0), rope_scaling.get("attention_factor", 1.0)
        )
        if end / orig_max > 1.0:
            inv_dim = lambda b: (dim * math.log(orig_max / (b * 2 * math.pi))) / (2 * math.log(rope_base))
            low, high = max(math.floor(inv_dim(beta_fast)), 0), min(math.ceil(inv_dim(beta_slow)), dim // 2 - 1)
            ramp = torch.clamp((torch.arange(dim // 2, device=freqs.device).float() - low) / max(high - low, 0.001), 0, 1)
            freqs = freqs * (1 - ramp + ramp / factor)
    t = torch.arange(end, device=freqs.device)
    freqs = torch.outer(t, freqs).float()
    freqs_cos = torch.cat([torch.cos(freqs), torch.cos(freqs)], dim=-1) * attn_factor
    freqs_sin = torch.cat([torch.sin(freqs), torch.sin(freqs)], dim=-1) * attn_factor
    return freqs_cos, freqs_sin
```

```python
def apply_rotary_pos_emb(q, k, cos, sin, unsqueeze_dim=1):
    def rotate_half(x): return torch.cat((-x[..., x.shape[-1] // 2:], x[..., : x.shape[-1] // 2]), dim=-1)
    q_embed = ((q * cos.unsqueeze(unsqueeze_dim)) + (rotate_half(q) * sin.unsqueeze(unsqueeze_dim))).to(q.dtype)
    k_embed = ((k * cos.unsqueeze(unsqueeze_dim)) + (rotate_half(k) * sin.unsqueeze(unsqueeze_dim))).to(k.dtype)
    return q_embed, k_embed
```

```python
def repeat_kv(x: torch.Tensor, n_rep: int) -> torch.Tensor:
    bs, slen, num_key_value_heads, head_dim = x.shape
    if n_rep == 1: return x
    return (x[:, :, :, None, :].expand(bs, slen, num_key_value_heads, n_rep, head_dim).reshape(bs, slen, num_key_value_heads * n_rep, head_dim))
```

```python
class Attention(nn.Module):
    def __init__(self, config: MiniMindConfig):
        super().__init__()
        self.num_key_value_heads = config.num_attention_heads if config.num_key_value_heads is None else config.num_key_value_heads
        self.n_local_heads = config.num_attention_heads
        self.n_local_kv_heads = self.num_key_value_heads
        self.n_rep = self.n_local_heads // self.n_local_kv_heads
        self.head_dim = config.head_dim
        self.is_causal = True
        self.q_proj = nn.Linear(config.hidden_size, config.num_attention_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(config.hidden_size, self.num_key_value_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.hidden_size, self.num_key_value_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(config.num_attention_heads * self.head_dim, config.hidden_size, bias=False)
        self.q_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)
        self.k_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.dropout = config.dropout
        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention') and config.flash_attn

    def forward(self, x, position_embeddings, past_key_value=None, use_cache=False, attention_mask=None):
        bsz, seq_len, _ = x.shape
        xq, xk, xv = self.q_proj(x), self.k_proj(x), self.v_proj(x)
        xq = xq.view(bsz, seq_len, self.n_local_heads, self.head_dim)
        xk = xk.view(bsz, seq_len, self.n_local_kv_heads, self.head_dim)
        xv = xv.view(bsz, seq_len, self.n_local_kv_heads, self.head_dim)
        xq, xk = self.q_norm(xq), self.k_norm(xk)
        cos, sin = position_embeddings
        xq, xk = apply_rotary_pos_emb(xq, xk, cos, sin)
        if past_key_value is not None:
            xk = torch.cat([past_key_value[0], xk], dim=1)
            xv = torch.cat([past_key_value[1], xv], dim=1)
        past_kv = (xk, xv) if use_cache else None
        xq, xk, xv = (xq.transpose(1, 2), repeat_kv(xk, self.n_rep).transpose(1, 2), repeat_kv(xv, self.n_rep).transpose(1, 2))
        if self.flash and (seq_len > 1) and (not self.is_causal or past_key_value is None) and (attention_mask is None or torch.all(attention_mask == 1)):
            output = F.scaled_dot_product_attention(xq, xk, xv, dropout_p=self.dropout if self.training else 0.0, is_causal=self.is_causal)
        else:
            scores = (xq @ xk.transpose(-2, -1)) / math.sqrt(self.head_dim)
            if self.is_causal: scores[:, :, :, -seq_len:] += torch.full((seq_len, seq_len), float("-inf"), device=scores.device).triu(1)
            if attention_mask is not None: scores += (1.0 - attention_mask.unsqueeze(1).unsqueeze(2)) * -1e9
            output = self.attn_dropout(F.softmax(scores.float(), dim=-1).type_as(xq)) @ xv
        output = output.transpose(1, 2).reshape(bsz, seq_len, -1)
        output = self.resid_dropout(self.o_proj(output))
        return output, past_kv
```

In [5]:
# 调用 src 中的可复用检查工具，比较 GQA 与 MHA
inspection_device = "cuda" if torch.cuda.is_available() else "cpu"
result = inspect_minimind(source_dir, seed=2026, device=inspection_device)
for key in ["gqa", "mha"]:
    item = result[key]
    print(f"\n{item['attention_type']}")
    print("参数量:", f"{item['parameter_count']:,}")
    print("设备:", item["device"], item["device_name"])
    print("dtype:", item["dtype"])
    print("hook 观察到的 Q shape:", item["query_shape_before_transpose"])
    print("hook 观察到的 K shape:", item["key_shape_before_repeat"])
    print("hook 观察到的 V shape:", item["value_shape_before_repeat"])
    print("四层输出:", item["layer_output_shapes"])
    print("logits shape:", item["logits_shape"])


GQA
参数量: 4,983,552
设备: cuda:0 NVIDIA GeForce RTX 4060 Laptop GPU
dtype: torch.float32
hook 观察到的 Q shape: [2, 8, 4, 64]
hook 观察到的 K shape: [2, 8, 2, 64]
hook 观察到的 V shape: [2, 8, 2, 64]
四层输出: {'layer_0': [2, 8, 256], 'layer_1': [2, 8, 256], 'layer_2': [2, 8, 256], 'layer_3': [2, 8, 256]}
logits shape: [2, 8, 6400]

MHA
参数量: 5,245,696
设备: cuda:0 NVIDIA GeForce RTX 4060 Laptop GPU
dtype: torch.float32
hook 观察到的 Q shape: [2, 8, 4, 64]
hook 观察到的 K shape: [2, 8, 4, 64]
hook 观察到的 V shape: [2, 8, 4, 64]
四层输出: {'layer_0': [2, 8, 256], 'layer_1': [2, 8, 256], 'layer_2': [2, 8, 256], 'layer_3': [2, 8, 256]}
logits shape: [2, 8, 6400]


## 4. KV Cache：复用历史 token 的 K/V

不使用 cache 时，每生成一个新 token 都要重新计算整个前缀的 Q/K/V。使用 cache 时，历史 K/V 被保存；下一步只为新 token 计算 Q/K/V，再把新的 K/V 接到历史 cache 后面。

KV Cache 不改变模型定义的预测目标。它改变的是自回归推理时避免重复计算的方式。若同时计算 key 与 value，单层 cache 的元素数为

$$
N_{\mathrm{cache}}=2BTH_{\mathrm{kv}}d_{\mathrm{head}}.
$$

因此 cache 会随 $B$、$T$、$H_{\mathrm{kv}}$ 和 $d_{\mathrm{head}}$ 线性增长，用额外显存换取更少的重复计算。相同 $B$、$T$、$d_{\mathrm{head}}$ 和 query head 数下，MHA 满足 $H_{\mathrm{kv}}=H_{\mathrm q}$，所以

$$
\frac{N_{\mathrm{cache,GQA}}}{N_{\mathrm{cache,MHA}}}
=\frac{H_{\mathrm{kv}}}{H_{\mathrm q}}.
$$

若每 $g=H_{\mathrm q}/H_{\mathrm{kv}}$ 个 query heads 共享一组 K/V，GQA cache 就是 MHA 的 $1/g$。这里的 $g$ 是共享组大小，不是 KV head 数。本次配置中 $H_{\mathrm q}=4$、$H_{\mathrm{kv}}=2$，比例为 $1/2$。

MiniMind 模型本身已经实现 KV Cache。本阶段直接调用它即可，不需要部署 vLLM 等推理服务。

下面的检查同时使用 `model.eval()` 和 `torch.no_grad()`。前者关闭 dropout 等训练模式行为；后者不记录梯度和计算图。

In [6]:
# 查看 cache 的长度增长，并核对增量与完整 forward 的最后位置 logits
for key in ["gqa", "mha"]:
    item = result[key]
    print(f"\n{item['attention_type']}")
    print("关闭 cache 后四层是否均为 None:", item["cache_disabled_entries_are_none"])
    print("7-token prefix K:", item["prefix_key_shape"])
    print("增加 1 token 后 K:", item["incremental_key_shape"])
    print("cache/full 最大绝对差:", item["cache_vs_full_last_logit_max_abs_diff"])
batch_size, sequence_length, head_dim = 2, 8, 64
gqa_kv_heads = result["gqa"]["key_shape_before_repeat"][2]
mha_kv_heads = result["mha"]["key_shape_before_repeat"][2]
gqa_kv_cache_elements = 2 * batch_size * sequence_length * gqa_kv_heads * head_dim
mha_kv_cache_elements = 2 * batch_size * sequence_length * mha_kv_heads * head_dim
print("\nGQA/MHA 的 KV head 比例:", gqa_kv_heads / mha_kv_heads)
print("单层 K+V cache 元素数，GQA:", gqa_kv_cache_elements)
print("单层 K+V cache 元素数，MHA:", mha_kv_cache_elements)
print("单层 K+V cache 比例，GQA/MHA:", gqa_kv_cache_elements / mha_kv_cache_elements)


GQA
关闭 cache 后四层是否均为 None: [True, True, True, True]
7-token prefix K: [2, 7, 2, 64]
增加 1 token 后 K: [2, 8, 2, 64]
cache/full 最大绝对差: 5.364418029785156e-07

MHA
关闭 cache 后四层是否均为 None: [True, True, True, True]
7-token prefix K: [2, 7, 4, 64]
增加 1 token 后 K: [2, 8, 4, 64]
cache/full 最大绝对差: 4.76837158203125e-07

GQA/MHA 的 KV head 比例: 0.5
单层 K+V cache 元素数，GQA: 4096
单层 K+V cache 元素数，MHA: 8192
单层 K+V cache 比例，GQA/MHA: 0.5


## 5. gated FFN 与 MiniMindBlock

FeedForward 的输入和输出均属于 $\mathbb{R}^{B\times T\times D}$。对单个 hidden vector $\mathbf{x}$，Dense gated FFN 计算

$$
\operatorname{FFN}(\mathbf{x})
=W_{\mathrm{down}}\!\left(
\operatorname{SiLU}(W_{\mathrm{gate}}\mathbf{x})
\odot W_{\mathrm{up}}\mathbf{x}
\right).
$$

`gate_proj` 和 `up_proj` 对应两个升维分支，`down_proj` 将逐元素乘积投影回维度 $D$。`hidden_act='silu'` 指定门控分支的激活函数；`SiLU(gate_proj(x)) * up_proj(x)` 这一整体是 SwiGLU 风格的门控 FFN。

MiniMindBlock 使用 Pre-Norm。令 $\mathbf{x}$ 为 block 输入，代码的准确数据流为

$$
\mathbf{x}_{\mathrm{attn}}=\mathbf{x}+\operatorname{Attention}(\operatorname{RMSNorm}(\mathbf{x})),
$$

$$
\mathbf{x}_{\mathrm{out}}=\mathbf{x}_{\mathrm{attn}}+\operatorname{MLP}(\operatorname{RMSNorm}(\mathbf{x}_{\mathrm{attn}})).
$$

```text
原始 hidden_states
       │
       ├─────────────── residual ─────────────┐
       │                                      │
       └→ RMSNorm → Attention → attention_output
                                              │
                     attention_residual = residual + attention_output
                                              │
                  ┌───────────────────────────┴──────────────┐
                  │                                          │
                  └→ RMSNorm → MLP → mlp_output              │
                                                             │
                  最终结果 = attention_residual + mlp_output ┘
```

第二次 RMSNorm 在第一次 residual 之后、MLP 之前执行，所以它是 MLP 子层的 Pre-Norm。归一化结果只进入 MLP 分支；`attention_residual` 仍作为未被覆盖的残差主干继续相加。各 block 会持续更新残差流，`MiniMindModel` 在全部 block 之后再执行 final RMSNorm。

In [7]:
# 阅读 Dense FFN 与一个完整 block
show_source(inspect.getsource(model_module.FeedForward))
show_source(inspect.getsource(model_module.MiniMindBlock))

```python
class FeedForward(nn.Module):
    def __init__(self, config: MiniMindConfig, intermediate_size: int = None):
        super().__init__()
        intermediate_size = intermediate_size or config.intermediate_size
        self.gate_proj = nn.Linear(config.hidden_size, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, config.hidden_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, intermediate_size, bias=False)
        self.act_fn = ACT2FN[config.hidden_act]

    def forward(self, x):
        return self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))
```

```python
class MiniMindBlock(nn.Module):
    def __init__(self, layer_id: int, config: MiniMindConfig):
        super().__init__()
        self.self_attn = Attention(config)
        self.input_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.post_attention_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.mlp = FeedForward(config) if not config.use_moe else MOEFeedForward(config)

    def forward(self, hidden_states, position_embeddings, past_key_value=None, use_cache=False, attention_mask=None):
        residual = hidden_states
        hidden_states, present_key_value = self.self_attn(
            self.input_layernorm(hidden_states), position_embeddings,
            past_key_value, use_cache, attention_mask
        )
        hidden_states += residual
        hidden_states = hidden_states + self.mlp(self.post_attention_layernorm(hidden_states))
        return hidden_states, present_key_value
```

## 6. Model、LM Head 与 loss

`MiniMindModel` 负责 embedding、RoPE buffer、逐层 block、final RMSNorm 和 cache 列表。输入 token IDs 的 shape 为 $B\times T$。`nn.Embedding(V, D)` 保存一张 $V\times D$ 的表，并按每个 token ID 取出对应行，得到 $\mathbf{H}^{(0)}\in\mathbb{R}^{B\times T\times D}$。实际实现是索引查表，不会构造 one-hot Tensor；概念上可把它理解为 one-hot 与 embedding 矩阵相乘。

所有 block 和 final RMSNorm 输出 $\mathbf{H}\in\mathbb{R}^{B\times T\times D}$。`nn.Linear(D, V, bias=False)` 的概念映射是 $D\rightarrow V$，PyTorch 将其权重存为 $\mathbf{W}\in\mathbb{R}^{V\times D}$，并计算 $\mathbf{Z}=\mathbf{H}\mathbf{W}^{\mathsf T}$，得到 $\mathbf{Z}\in\mathbb{R}^{B\times T\times V}$。训练对每个位置计算 next-token logits；生成时通常只取 `logits[:, -1, :]`，其 shape 为 $B\times V$。

Embedding 与 LM head 功能不同，但二者都为同一词表中的每个 token 保留一个 $D$ 维行向量，而且 PyTorch 中两份权重的存储 shape 都是 $V\times D$。官方实现默认让 `model.embed_tokens.weight` 与 `lm_head.weight` 指向同一个 Parameter：输入侧把该行作为 token 表示，输出侧用同一行与 hidden state 计算匹配分数。权重绑定是一种可选的参数共享设计，不是语言模型必须满足的数学条件；当前配置因此少保存一份包含 $VD$ 个参数的矩阵。

计算 loss 时，logits 去掉最后一个位置，labels 去掉第一个位置，从而对齐 next-token prediction：

$$
\mathcal{L}
=-\frac{1}{N_{\mathrm{valid}}}
\sum_{(b,t)\in\mathcal{I}_{\mathrm{valid}}}
\log p_\theta(x_{b,t+1}\mid x_{b,\le t}).
$$

标签值为 $-100$ 的位置不属于有效位置集合 $\mathcal{I}_{\mathrm{valid}}$，不参与 cross entropy。

In [8]:
# 阅读模型主干和完整 causal LM 类，包括权重绑定、loss 与生成时的 cache 更新
show_source(inspect.getsource(model_module.MiniMindModel))
show_source(inspect.getsource(model_module.MiniMindForCausalLM))

```python
class MiniMindModel(nn.Module):
    def __init__(self, config: MiniMindConfig):
        super().__init__()
        self.config = config
        self.vocab_size, self.num_hidden_layers = config.vocab_size, config.num_hidden_layers
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.dropout = nn.Dropout(config.dropout)
        self.layers = nn.ModuleList([MiniMindBlock(l, config) for l in range(self.num_hidden_layers)])
        self.norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        freqs_cos, freqs_sin = precompute_freqs_cis(dim=config.head_dim, end=config.max_position_embeddings, rope_base=config.rope_theta, rope_scaling=config.rope_scaling)
        self.register_buffer("freqs_cos", freqs_cos, persistent=False)
        self.register_buffer("freqs_sin", freqs_sin, persistent=False)

    def forward(self, input_ids, attention_mask=None, past_key_values=None, use_cache=False, **kwargs):
        batch_size, seq_length = input_ids.shape
        if hasattr(past_key_values, 'layers'): past_key_values = None
        past_key_values = past_key_values or [None] * len(self.layers)
        start_pos = past_key_values[0][0].shape[1] if past_key_values[0] is not None else 0
        hidden_states = self.dropout(self.embed_tokens(input_ids))
        # Recompute RoPE buffers lost during meta-device init (transformers>=5.x)
        if self.freqs_cos[0, 0] == 0:
            freqs_cos, freqs_sin = precompute_freqs_cis(dim=self.config.head_dim, end=self.config.max_position_embeddings, rope_base=self.config.rope_theta, rope_scaling=self.config.rope_scaling)
            self.freqs_cos, self.freqs_sin = freqs_cos.to(hidden_states.device), freqs_sin.to(hidden_states.device)
        position_embeddings = (self.freqs_cos[start_pos:start_pos + seq_length], self.freqs_sin[start_pos:start_pos + seq_length])
        presents = []
        for layer, past_key_value in zip(self.layers, past_key_values):
            hidden_states, present = layer(
                hidden_states,
                position_embeddings,
                past_key_value=past_key_value,
                use_cache=use_cache,
                attention_mask=attention_mask
            )
            presents.append(present)
        hidden_states = self.norm(hidden_states)
        aux_loss = sum([l.mlp.aux_loss for l in self.layers if isinstance(l.mlp, MOEFeedForward)], hidden_states.new_zeros(1).squeeze())
        return hidden_states, presents, aux_loss
```

```python
class MiniMindForCausalLM(PreTrainedModel, GenerationMixin):
    config_class = MiniMindConfig
    _tied_weights_keys = {"lm_head.weight": "model.embed_tokens.weight"}
    def __init__(self, config: MiniMindConfig = None):
        self.config = config or MiniMindConfig()
        super().__init__(self.config)
        self.model = MiniMindModel(self.config)
        self.lm_head = nn.Linear(self.config.hidden_size, self.config.vocab_size, bias=False)
        if self.config.tie_word_embeddings: self.model.embed_tokens.weight = self.lm_head.weight
        self.post_init()

    def forward(self, input_ids, attention_mask=None, past_key_values=None, use_cache=False, logits_to_keep=0, labels=None, **kwargs):
        hidden_states, past_key_values, aux_loss = self.model(input_ids, attention_mask, past_key_values, use_cache, **kwargs)
        slice_indices = slice(-logits_to_keep, None) if isinstance(logits_to_keep, int) else logits_to_keep
        logits = self.lm_head(hidden_states[:, slice_indices, :])
        loss = None
        if labels is not None:
            x, y = logits[..., :-1, :].contiguous(), labels[..., 1:].contiguous()
            loss = F.cross_entropy(x.view(-1, x.size(-1)), y.view(-1), ignore_index=-100)
        return MoeCausalLMOutputWithPast(loss=loss, aux_loss=aux_loss, logits=logits, past_key_values=past_key_values, hidden_states=hidden_states)
    
    # https://github.com/jingyaogong/minimind/discussions/611
    @torch.inference_mode()
    def generate(self, inputs=None, attention_mask=None, max_new_tokens=8192, temperature=0.85, top_p=0.85, top_k=50, eos_token_id=2, streamer=None, use_cache=True, num_return_sequences=1, do_sample=True, repetition_penalty=1.0, **kwargs):
        input_ids = kwargs.pop("input_ids", inputs).repeat(num_return_sequences, 1)
        attention_mask = attention_mask.repeat(num_return_sequences, 1) if attention_mask is not None else None
        past_key_values = kwargs.pop("past_key_values", None)
        finished = torch.zeros(input_ids.shape[0], dtype=torch.bool, device=input_ids.device)
        if streamer: streamer.put(input_ids.cpu())
        for _ in range(max_new_tokens):
            past_len = past_key_values[0][0].shape[1] if past_key_values else 0
            outputs = self.forward(input_ids[:, past_len:], attention_mask, past_key_values, use_cache=use_cache, **kwargs)
            attention_mask = torch.cat([attention_mask, attention_mask.new_ones(attention_mask.shape[0], 1)], -1) if attention_mask is not None else None
            logits = outputs.logits[:, -1, :] / temperature
            if repetition_penalty != 1.0:
                for i in range(input_ids.shape[0]):
                    seen = torch.unique(input_ids[i]); score = logits[i, seen]; logits[i, seen] = torch.where(score > 0, score / repetition_penalty, score * repetition_penalty)
            if top_k > 0: 
                logits[logits < torch.topk(logits, top_k)[0][..., -1, None]] = -float('inf')
            if top_p < 1.0:
                sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                mask = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1) > top_p
                mask[..., 1:], mask[..., 0] = mask[..., :-1].clone(), 0
                logits[mask.scatter(1, sorted_indices, mask)] = -float('inf')
            next_token = torch.multinomial(torch.softmax(logits, dim=-1), num_samples=1) if do_sample else torch.argmax(logits, dim=-1, keepdim=True)
            if eos_token_id is not None: next_token = torch.where(finished.unsqueeze(-1), next_token.new_full((next_token.shape[0], 1), eos_token_id), next_token)
            input_ids = torch.cat([input_ids, next_token], dim=-1)
            past_key_values = outputs.past_key_values if use_cache else None
            if streamer: streamer.put(next_token.cpu())
            if eos_token_id is not None:
                finished |= next_token.squeeze(-1).eq(eos_token_id)
                if finished.all(): break
        if streamer: streamer.end()
        if kwargs.get("return_kv"): return {'generated_ids': input_ids, 'past_kv': past_key_values}
        return input_ids
```

## 7. PretrainDataset 与训练入口

`PretrainDataset` 把一条文本编码为 token 序列 $([\mathrm{BOS}],x_1,\ldots,x_n,[\mathrm{EOS}])$，再 right padding 到固定长度。labels 初始复制 input IDs，padding 位置改为 $-100$。模型内部负责 shift。

官方 Dataset 不返回 padding `attention_mask`。对当前 right-padding 排列，causal mask 会阻止前面的有效 token 看到未来 padding，且 padding labels 为 $-100$，所以 padding 位置不计入 loss；padding query 的计算仍然存在。left padding、包含其他结构边界的序列或带 KV Cache 的批量生成需要正确传入额外 mask。

训练循环依次执行 forward、除以 gradient accumulation steps、backward、gradient clipping、optimizer step、日志与 checkpoint。`AdamW` 负责根据梯度更新参数，`get_lr` 只负责计算当前学习率。阶段 4 会实际修改和验证这条训练链路；当前阶段先识别官方实现的准确行为和待验证边界。

In [9]:
# 加载并展示官方 Dataset；训练脚本展示核心训练循环和初始化段
dataset_path = source_dir / "dataset" / "lm_dataset.py"
dataset_spec = importlib.util.spec_from_file_location("stage3_lm_dataset", dataset_path)
dataset_module = importlib.util.module_from_spec(dataset_spec)
sys.modules[dataset_spec.name] = dataset_module
dataset_spec.loader.exec_module(dataset_module)
show_source(inspect.getsource(dataset_module.PretrainDataset))

train_path = source_dir / "trainer" / "train_pretrain.py"
train_lines = train_path.read_text(encoding="utf-8").splitlines()
for start, end, title in [(24, 80, "完整 train_epoch"), (83, len(train_lines), "完整训练入口")]:
    print(f"\n--- {title}: lines {start}-{end} ---")
    numbered_source = "\n".join(
        f"{number:3d}: {train_lines[number - 1]}"
        for number in range(start, min(end, len(train_lines)) + 1)
    )
    show_source(numbered_source)

trainer_utils_path = source_dir / "trainer" / "trainer_utils.py"
trainer_utils_lines = trainer_utils_path.read_text(encoding="utf-8").splitlines()
for start, end, title in [(40, 41, "get_lr"), (63, 116, "lm_checkpoint")]:
    print(f"\n--- {title}: trainer_utils.py lines {start}-{end} ---")
    numbered_source = "\n".join(
        f"{number:3d}: {trainer_utils_lines[number - 1]}"
        for number in range(start, min(end, len(trainer_utils_lines)) + 1)
    )
    show_source(numbered_source)

```python
class PretrainDataset(Dataset):
    def __init__(self, data_path, tokenizer, max_length=512):
        super().__init__()
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.samples = load_dataset('json', data_files=data_path, split='train')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        tokens = self.tokenizer(str(sample['text']), add_special_tokens=False, max_length=self.max_length - 2, truncation=True).input_ids
        tokens = [self.tokenizer.bos_token_id] + tokens + [self.tokenizer.eos_token_id]
        input_ids = tokens + [self.tokenizer.pad_token_id] * (self.max_length - len(tokens))
        input_ids = torch.tensor(input_ids, dtype=torch.long)
        labels = input_ids.clone()
        labels[input_ids == self.tokenizer.pad_token_id] = -100
        return input_ids, labels
```


--- 完整 train_epoch: lines 24-80 ---


```python
 24: def train_epoch(epoch, loader, iters, start_step=0, wandb=None):
 25:     start_time = time.time()
 26:     last_step = start_step
 27:     for step, (input_ids, labels) in enumerate(loader, start=start_step + 1):
 28:         input_ids = input_ids.to(args.device)
 29:         labels = labels.to(args.device)
 30:         last_step = step
 31:         lr = get_lr(epoch * iters + step, args.epochs * iters, args.learning_rate)
 32:         for param_group in optimizer.param_groups:
 33:             param_group['lr'] = lr
 34: 
 35:         with autocast_ctx:
 36:             res = model(input_ids, labels=labels)
 37:             loss = res.loss + res.aux_loss
 38:             loss = loss / args.accumulation_steps
 39: 
 40:         scaler.scale(loss).backward()
 41: 
 42:         if step % args.accumulation_steps == 0:
 43:             scaler.unscale_(optimizer)
 44:             torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
 45: 
 46:             scaler.step(optimizer)
 47:             scaler.update()
 48: 
 49:             optimizer.zero_grad(set_to_none=True)
 50: 
 51:         if step % args.log_interval == 0 or step == iters:
 52:             spend_time = time.time() - start_time
 53:             current_loss = loss.item() * args.accumulation_steps
 54:             current_aux_loss = res.aux_loss.item() if res.aux_loss is not None else 0.0
 55:             current_logits_loss = current_loss - current_aux_loss
 56:             current_lr = optimizer.param_groups[-1]['lr']
 57:             eta_min = spend_time / max(step - start_step, 1) * (iters - step) // 60
 58:             Logger(f'Epoch:[{epoch + 1}/{args.epochs}]({step}/{iters}), loss: {current_loss:.4f}, logits_loss: {current_logits_loss:.4f}, aux_loss: {current_aux_loss:.4f}, lr: {current_lr:.8f}, epoch_time: {eta_min:.1f}min')
 59:             if wandb: wandb.log({"loss": current_loss, "logits_loss": current_logits_loss, "aux_loss": current_aux_loss, "learning_rate": current_lr, "epoch_time": eta_min})
 60: 
 61:         if (step % args.save_interval == 0 or step == iters) and is_main_process():
 62:             model.eval()
 63:             moe_suffix = '_moe' if lm_config.use_moe else ''
 64:             ckp = f'{args.save_dir}/{args.save_weight}_{lm_config.hidden_size}{moe_suffix}.pth'
 65:             raw_model = model.module if isinstance(model, DistributedDataParallel) else model
 66:             raw_model = getattr(raw_model, '_orig_mod', raw_model)
 67:             state_dict = raw_model.state_dict()
 68:             torch.save({k: v.half().cpu() for k, v in state_dict.items()}, ckp)
 69:             lm_checkpoint(lm_config, weight=args.save_weight, model=model, optimizer=optimizer, scaler=scaler, epoch=epoch, step=step, wandb=wandb, save_dir='../checkpoints')
 70:             model.train()
 71:             del state_dict
 72: 
 73:         del input_ids, labels, res, loss
 74: 
 75:     if last_step > start_step and last_step % args.accumulation_steps != 0:
 76:         scaler.unscale_(optimizer)
 77:         torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
 78:         scaler.step(optimizer)
 79:         scaler.update()
 80:         optimizer.zero_grad(set_to_none=True)
```


--- 完整训练入口: lines 83-172 ---


```python
 83: if __name__ == "__main__":
 84:     parser = argparse.ArgumentParser(description="MiniMind Pretraining")
 85:     parser.add_argument("--save_dir", type=str, default="../out", help="模型保存目录")
 86:     parser.add_argument('--save_weight', default='pretrain', type=str, help="保存权重的前缀名")
 87:     parser.add_argument("--epochs", type=int, default=2, help="训练轮数")
 88:     parser.add_argument("--batch_size", type=int, default=32, help="batch size")
 89:     parser.add_argument("--learning_rate", type=float, default=5e-4, help="初始学习率")
 90:     parser.add_argument("--device", type=str, default="cuda:0" if torch.cuda.is_available() else "cpu", help="训练设备")
 91:     parser.add_argument("--dtype", type=str, default="bfloat16", help="混合精度类型")
 92:     parser.add_argument("--num_workers", type=int, default=8, help="数据加载线程数")
 93:     parser.add_argument("--accumulation_steps", type=int, default=8, help="梯度累积步数")
 94:     parser.add_argument("--grad_clip", type=float, default=1.0, help="梯度裁剪阈值")
 95:     parser.add_argument("--log_interval", type=int, default=100, help="日志打印间隔")
 96:     parser.add_argument("--save_interval", type=int, default=1000, help="模型保存间隔")
 97:     parser.add_argument('--hidden_size', default=768, type=int, help="隐藏层维度")
 98:     parser.add_argument('--num_hidden_layers', default=8, type=int, help="隐藏层数量")
 99:     parser.add_argument('--max_seq_len', default=340, type=int, help="训练的最大截断长度（中文1token≈1.5~1.7字符）")
100:     parser.add_argument('--use_moe', default=0, type=int, choices=[0, 1], help="是否使用MoE架构（0=否，1=是）")
101:     parser.add_argument("--data_path", type=str, default="../dataset/pretrain_t2t_mini.jsonl", help="预训练数据路径")
102:     parser.add_argument('--from_weight', default='none', type=str, help="基于哪个权重训练，为none则从头开始")
103:     parser.add_argument('--from_resume', default=0, type=int, choices=[0, 1], help="是否自动检测&续训（0=否，1=是）")
104:     parser.add_argument("--use_wandb", action="store_true", help="是否使用wandb")
105:     parser.add_argument("--wandb_project", type=str, default="MiniMind-Pretrain", help="wandb项目名")
106:     parser.add_argument("--use_compile", default=0, type=int, choices=[0, 1], help="是否使用torch.compile加速（0=否，1=是）")
107:     args = parser.parse_args()
108: 
109:     # ========== 1. 初始化环境和随机种子 ==========
110:     local_rank = init_distributed_mode()
111:     if dist.is_initialized(): args.device = f"cuda:{local_rank}"
112:     setup_seed(42 + (dist.get_rank() if dist.is_initialized() else 0))
113:     
114:     # ========== 2. 配置目录、模型参数、检查ckp ==========
115:     os.makedirs(args.save_dir, exist_ok=True)
116:     lm_config = MiniMindConfig(hidden_size=args.hidden_size, num_hidden_layers=args.num_hidden_layers, use_moe=bool(args.use_moe))
117:     ckp_data = lm_checkpoint(lm_config, weight=args.save_weight, save_dir='../checkpoints') if args.from_resume==1 else None
118:     
119:     # ========== 3. 设置混合精度 ==========
120:     device_type = "cuda" if "cuda" in args.device else "cpu"
121:     dtype = torch.bfloat16 if args.dtype == "bfloat16" else torch.float16
122:     autocast_ctx = nullcontext() if device_type == "cpu" else torch.cuda.amp.autocast(dtype=dtype)
123:     
124:     # ========== 4. 配wandb ==========
125:     wandb = None
126:     if args.use_wandb and is_main_process():
127:         import swanlab as wandb
128:         wandb_id = ckp_data.get('wandb_id') if ckp_data else None
129:         resume = 'must' if wandb_id else None
130:         wandb_run_name = f"MiniMind-Pretrain-Epoch-{args.epochs}-BatchSize-{args.batch_size}-LearningRate-{args.learning_rate}"
131:         wandb.init(project=args.wandb_project, name=wandb_run_name, id=wandb_id, resume=resume)
132:     
133:     # ========== 5. 定义模型、数据、优化器 ==========
134:     model, tokenizer = init_model(lm_config, args.from_weight, device=args.device)
135:     train_ds = PretrainDataset(args.data_path, tokenizer, max_length=args.max_seq_len)
136:     train_sampler = DistributedSampler(train_ds) if dist.is_initialized() else None
137:     scaler = torch.cuda.amp.GradScaler(enabled=(args.dtype == 'float16'))
138:     optimizer = optim.AdamW(model.parameters(), lr=args.learning_rate)
139:     
140:     # ========== 6. 从ckp恢复状态 ==========
141:     start_epoch, start_step = 0, 0
142:     if ckp_data:
143:         model.load_state_dict(ckp_data['model'])
144:         optimizer.load_state_dict(ckp_data['optimizer'])
145:         scaler.load_state_dict(ckp_data['scaler'])
146:         start_epoch = ckp_data['epoch']
147:         start_step = ckp_data.get('step', 0)
148:     
149:     # ========== 7. 编译和分布式包装 ==========
150:     if args.use_compile == 1:
151:         model = torch.compile(model)
152:         Logger('torch.compile enabled')
153:     if dist.is_initialized():
154:         model = DistributedDataParallel(model, device_ids=[local_rank])
155:     
156:     # ========== 8. 开始训练 ==========
157:     for epoch in range(start_epoch, args.epochs):
158:         train_sampler and train_sampler.set_epoch(epoch)
159:         setup_seed(42 + epoch); indices = torch.randperm(len(train_ds)).tolist()
160:         skip = start_step if (epoch == start_epoch and start_step > 0) else 0
161:         batch_sampler = SkipBatchSampler(train_sampler or indices, args.batch_size, skip)
162:         loader = DataLoader(train_ds, batch_sampler=batch_sampler, num_workers=args.num_workers, pin_memory=True)
163:         if skip > 0: 
164:             Logger(f'Epoch [{epoch + 1}/{args.epochs}]: 跳过前{start_step}个step，从step {start_step + 1}开始')
165:             train_epoch(epoch, loader, len(loader) + skip, start_step, wandb)
166:         else:
167:             train_epoch(epoch, loader, len(loader), 0, wandb)
168:     
169:     # ========== 9. 清理分布进程 ==========
170:     if dist.is_initialized():
171:         dist.barrier()
172:         dist.destroy_process_group()
```


--- get_lr: trainer_utils.py lines 40-41 ---


```python
 40: def get_lr(current_step, total_steps, lr):
 41:     return lr*(0.1 + 0.45*(1 + math.cos(math.pi * current_step / total_steps)))
```


--- lm_checkpoint: trainer_utils.py lines 63-116 ---


```python
 63: def lm_checkpoint(lm_config, weight='full_sft', model=None, optimizer=None, epoch=0, step=0, wandb=None, save_dir='../checkpoints', **kwargs):
 64:     os.makedirs(save_dir, exist_ok=True)
 65:     moe_path = '_moe' if lm_config.use_moe else ''
 66:     ckp_path = f'{save_dir}/{weight}_{lm_config.hidden_size}{moe_path}.pth'
 67:     resume_path = f'{save_dir}/{weight}_{lm_config.hidden_size}{moe_path}_resume.pth'
 68: 
 69:     if model is not None:
 70:         raw_model = model.module if isinstance(model, DistributedDataParallel) else model
 71:         raw_model = getattr(raw_model, '_orig_mod', raw_model)
 72:         state_dict = raw_model.state_dict()
 73:         state_dict = {k: v.half().cpu() for k, v in state_dict.items()}
 74:         ckp_tmp = ckp_path + '.tmp'
 75:         torch.save(state_dict, ckp_tmp)
 76:         os.replace(ckp_tmp, ckp_path)
 77:         wandb_id = None
 78:         if wandb:
 79:             if hasattr(wandb, 'get_run'):
 80:                 run = wandb.get_run()
 81:                 wandb_id = getattr(run, 'id', None) if run else None
 82:             else:
 83:                 wandb_id = getattr(wandb, 'id', None)
 84: 
 85:         resume_data = {
 86:             'model': state_dict,
 87:             'optimizer': optimizer.state_dict(),
 88:             'epoch': epoch,
 89:             'step': step,
 90:             'world_size': dist.get_world_size() if dist.is_initialized() else 1,
 91:             'wandb_id': wandb_id
 92:         }
 93:         for key, value in kwargs.items():
 94:             if value is not None:
 95:                 if hasattr(value, 'state_dict'):
 96:                     raw_value = value.module if isinstance(value, DistributedDataParallel) else value
 97:                     raw_value = getattr(raw_value, '_orig_mod', raw_value)
 98:                     resume_data[key] = raw_value.state_dict()
 99:                 else:
100:                     resume_data[key] = value
101: 
102:         resume_tmp = resume_path + '.tmp'
103:         torch.save(resume_data, resume_tmp)
104:         os.replace(resume_tmp, resume_path)
105:         del state_dict, resume_data
106:         torch.cuda.empty_cache()
107:     else:  # 加载模式
108:         if os.path.exists(resume_path):
109:             ckp_data = torch.load(resume_path, map_location='cpu')
110:             saved_ws = ckp_data.get('world_size', 1)
111:             current_ws = dist.get_world_size() if dist.is_initialized() else 1
112:             if saved_ws != current_ws:
113:                 ckp_data['step'] = ckp_data['step'] * saved_ws // current_ws
114:                 Logger(f'GPU数量变化({saved_ws}→{current_ws})，step已自动转换为{ckp_data["step"]}')
115:             return ckp_data
116:         return None
```

### 7.1 学习率与梯度累积

官方代码使用 `optim.AdamW(model.parameters(), lr=args.learning_rate)` 创建优化器，再用 `get_lr` 手写余弦衰减。它从初始学习率平滑降到初始值的 10%，没有 warmup。训练循环在每个数据 batch 都计算并写入一次学习率，但只有满足 `step % accumulation_steps == 0` 时才调用 `optimizer.step()`。因此一个累积周期内，前几次学习率赋值没有参与参数更新，真正生效的是更新边界上的值。

如果只想消除重复赋值并保持当前实际更新行为，可以把 `get_lr` 和 `param_group['lr']` 移到参数更新分支，并放在 `scaler.step(optimizer)` 之前。如果希望余弦曲线严格按 AdamW 更新次数推进，则还要改用 `optimizer_step` 和 `total_optimizer_steps`。阶段 4 再选择实现并通过短训练验证。

### 7.2 checkpoint 的触发与内容

保存条件是 `(step % save_interval == 0 or step == iters) and is_main_process()`。默认每 1000 个数据 batch 保存一次，并在每个 epoch 的最后一个 batch 保存；DDP 中只有主进程写文件。`step` 统计数据 batch，不是 AdamW 更新次数。文件名不包含 step，因此后一次保存会覆盖前一次。

脚本保存两类内容：`save_dir` 与 `../checkpoints` 中的模型权重文件只包含转为 FP16/CPU 的 `state_dict`；`../checkpoints/*_resume.pth` 另外保存 optimizer、GradScaler、epoch、step、world size 和实验跟踪 ID，用于 `--from_resume 1`。当前 resume 文件没有保存项目 README 要求的完整 config、tokenizer 标识、指标历史和随机数状态，阶段 4 需要补齐并测试恢复一致性。

还要注意保存与梯度累积的顺序：如果保存 step 不在累积边界，尚未应用的梯度不会进入 checkpoint；epoch 最后不足一个完整累积周期时，源码先保存 checkpoint，再在循环外执行剩余的 `optimizer.step()`，因此该 epoch 末文件会漏掉最后一次参数更新。阶段 4 应把正式 checkpoint 放在参数更新完成之后。

## 8. 与 TinyGPT 对照

| 结构 | TinyGPT | MiniMind Dense | 作用 |
| --- | --- | --- | --- |
| 位置编码 | learned absolute embedding | RoPE | 将位置信息作用在 $\mathbf Q/\mathbf K$ 上 |
| normalization | LayerNorm | RMSNorm | 稳定子层输入尺度，省略均值中心化 |
| attention | MHA | GQA | 减少 $\mathbf K/\mathbf V$ 投影参数和 KV Cache |
| $\mathbf Q/\mathbf K$ | 无额外归一化 | QK-Norm | 控制 attention logits 数值尺度 |
| attention kernel | 手写 score/softmax | SDPA | 使用统一接口选择优化 kernel |
| MLP | GELU 两层 MLP | SwiGLU 风格 gated FFN | 用 SiLU 门控分支调节信息 |
| 增量推理 | 重算完整上下文 | 原生 KV Cache | 跨 forward 复用历史 $\mathbf K/\mathbf V$ |

这些结构不会改变语言模型的基本目标：token IDs 属于 $\mathbb{N}^{B\times T}$，输出 logits 属于 $\mathbb{R}^{B\times T\times V}$，loss 仍是错开一个 token 的 cross entropy。

## 复习题

**问题：GQA 为什么能缩小 KV Cache？**

答案：cache 保存的是复制前的 key/value。GQA 满足 $H_{\mathrm{kv}}<H_{\mathrm q}$，因此 $B\times T\times H_{\mathrm{kv}}\times d_{\mathrm{head}}$ 中的元素更少；计算 attention 前才用 `repeat_kv` 映射到 query heads。相同其他维度下，GQA/MHA cache 比例为 $H_{\mathrm{kv}}/H_{\mathrm q}$；若每 $g$ 个 query heads 共享一组 K/V，该比例也等于 $1/g$。

**问题：RoPE 为什么只作用于 Q 和 K？**

答案：attention 的位置相关性由 $\mathbf Q\mathbf K^{\mathsf T}$ 决定。旋转 query/key 会使点积包含相对位置关系，value 继续承载被加权汇总的内容。

**问题：为什么标准的整段并行 teacher-forcing 通常不使用推理式 KV Cache？**

答案：标准训练在一次 forward 中并行计算整段序列，每个位置的 K/V 本来就只计算一次，没有跨 forward 重算前缀的问题。分块训练可以缓存之前的 K/V；保留计算图时仍要保存反向传播所需状态，通常没有推理式 cache 的显存优势，断开计算图则会截断跨 chunk 梯度。

**问题：第二次 RMSNorm 明明位于第一次残差连接之后，为什么仍叫 Pre-Norm？**

答案：它位于下一个 MLP 子层之前，归一化结果只进入 MLP 分支。第一次残差结果仍作为主干直接参与第二次相加，没有被归一化结果覆盖。

**问题：MiniMind 增加这些现代结构后，语言模型目标是否改变？**

答案：没有。模型仍根据前缀预测下一个 token，结构变化主要影响位置表达、数值稳定性、参数使用和推理效率。

## 当前结论

在固定 revision、RTX 4060、`float32`、当前缩小配置和随机输入下，GQA 与 MHA 的 block 输出均属于 $\mathbb{R}^{B\times T\times D}$。本实验中 $H_{\mathrm{kv}}/H_{\mathrm q}=1/2$，所以 GQA 的 cache 元素数是 MHA 的一半，同时少 $262{,}144$ 个投影参数。完整 forward 与 cache 增量 forward 的最后位置 logits 最大绝对差记录在上方输出中。

这些结果验证了当前样例的 shape、cache 拼接和两条 forward 路径，不代表对全部输入、精度和设备的完整证明。完成本 Notebook 后，应脱离笔记重新画出 MiniMind Dense 的 forward，并逐项说明它相对 TinyGPT 增加的结构及其作用。

## 下一阶段前瞻：MiniMind 从头预训练

本节仿照阶段 0 的前瞻部分，只建立阶段 4 需要的知识框架，不在本 Notebook 中修改训练器或执行正式训练。阶段 4 的目标是把已经看懂的模型结构接入一条可验证、可评估、可恢复的训练链路。

### 1. 区分数据 step 与 optimizer step

梯度累积把一个参数更新拆成多个 micro-batch：

```text
读取 micro-batch
→ forward
→ loss / accumulation_steps
→ backward 并累积梯度
→ 达到累积边界
→ gradient clipping
→ AdamW 更新参数
→ 清空梯度
```

每读取一个 micro-batch，数据 step 增加一次；只有调用 `optimizer.step()`，optimizer step 才增加一次。若单卡 micro-batch size 为 $B_{\mathrm{micro}}$、梯度累积次数为 $A$、进程数为 $W$，按样本数计算的有效 batch size 为

$$
B_{\mathrm{effective}}=B_{\mathrm{micro}}AW.
$$

padding 不属于有效训练 token，因此每次更新实际使用的 token 数还要按非 `-100` 标签统计。学习率调度和 checkpoint 边界也需要明确使用哪一种 step。

### 2. 混合精度不是把所有状态都改成同一种 dtype

| 精度 | 主要特点 | 阶段 4 关注点 |
| --- | --- | --- |
| FP32 | 数值范围和精度较高，显存占用较大 | 作为小规模正确性对照 |
| FP16 | 数值范围较小 | 通常配合 GradScaler 防止梯度下溢 |
| BF16 | 数值范围接近 FP32、尾数精度较低 | MiniMind 默认训练 dtype，RTX 4060 上实际检查 |

autocast 控制部分算子的计算 dtype，模型参数、梯度和 AdamW 状态不一定全部采用同一 dtype。阶段 4 需要同时记录配置 dtype、实际设备、峰值显存和数值异常，而不是只看到 `bfloat16` 参数就认为整条训练链路都使用 BF16。

### 3. 先做固定小数据过拟合测试

正式训练前，阶段 4 先从固定 revision 的数据中按原始顺序取前 256 条，保存 Dataset fingerprint 与 row ID。缩小模型应在这组数据上明显降低 train loss。这个实验检查的是训练链路，不评价生成质量。

若无法过拟合，应依次检查 tokenization、label shift、padding 的 `-100`、attention mask、loss 缩放、梯度、AdamW 和学习率。直接扩大模型、延长训练或更换数据不能替代这项正确性检查。

### 4. Validation loss 与 perplexity

训练 loss 只能说明模型对训练数据的拟合情况。Validation 使用未参与参数更新的数据，并同时启用 `model.eval()` 与 `torch.no_grad()`。前者切换 dropout 等模块行为，后者关闭计算图记录。

Perplexity 由平均 token loss 得到：

$$
\operatorname{PPL}=\exp(\mathcal{L}_{\mathrm{token}}).
$$

不同 batch 的有效 token 数可能不同，因此正式 validation 应累计有效 token 的 loss 总和，再除以有效 token 总数，不能直接对 batch loss 做无权平均。

### 5. Checkpoint 要证明能够恢复训练

项目中的训练 checkpoint 不只服务于推理。按照根 README，它需要保存模型、optimizer、step、配置、tokenizer、指标历史和随机数状态；使用 FP16 时还需要 GradScaler 状态。

阶段 4 会进行短 resume 对照：

```text
连续训练 N 个 optimizer steps
          对比
训练 K 步 → 保存 → 重新加载 → 训练 N-K 步
```

在固定数据顺序和随机状态下，需要比较 step、学习率、loss 与模型参数。文件能够加载，只能证明序列化格式可读，不能证明恢复结果一致。

### 6. 阶段 4 的最小监控面板

| 指标 | 回答的问题 |
| --- | --- |
| train loss | 模型是否正在拟合训练数据 |
| validation loss / perplexity | 未参与训练的数据表现如何 |
| learning rate | 调度是否按预期推进 |
| gradient norm | 梯度是否消失、爆炸或被频繁裁剪 |
| peak GPU memory | 当前 batch、序列长度和 dtype 是否适配设备 |
| tokens/s | 训练吞吐是否稳定 |
| trained tokens | 不同运行实际消耗的数据量是否可比较 |

### 7. 下一阶段的执行顺序

```text
修正并测试 optimizer-step 学习率与 checkpoint 边界
→ 固定 256 条数据完成过拟合测试
→ 增加 token-weighted validation 与 perplexity
→ 完成 checkpoint 短 resume 对照
→ 在 RTX 4060 上运行官方 64M Dense mini pretrain
→ 记录 loss、显存、吞吐和累计 token 数
```

标准整段 teacher-forcing 训练默认关闭 `use_cache`，因为单次 forward 中每个位置的 K/V 已经只计算一次；这描述的是阶段 4 的实现选择，不表示任何形式的 teacher-forcing 都禁止使用 cache。本阶段仍不进入 SFT、LoRA、DPO、RL 或 vLLM 部署。